[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1kCktQ6oXUZ0CDryXBjja4RyRfQuxFBwG/view?usp=drive_link)

# Agent Evaluation – Pre-Captured Traces

This notebook demonstrates how to evaluate agents when you already have conversation logs (traces). No agent runs during evaluation — Floeval scores the traces directly.

**Objectives**
- Install Floeval and configure credentials
- Load an `AgentDataset` with full traces from a JSON file you provide
- Configure `AgentEvaluation` with LLM-judge metrics
- Run the evaluation and inspect the summary

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
%pip install floeval>=0.2.0b1

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [1]:
import getpass
# LLM and API configuration (OpenAI)

OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("Enter your API key: ")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 3. Imports

Import the core Floeval classes used to load agent traces and run evaluation metrics.

In [2]:
from pathlib import Path

from floeval.api.agent_evaluation import AgentEvaluation
from floeval.api.dataset_loaders.agent_file_loader import AgentDatasetLoader
from floeval.config.schemas.io.llm import OpenAIProviderConfig

## 4. Load the Agent Dataset (JSON)

Minimal JSON shape (pre-captured **trace** per sample):

```json
{
  "samples": [
    {
      "user_input": "...",
      "trace": {
        "messages": [
          { "role": "human", "content": "..." },
          { "role": "ai", "content": "..." }
        ],
        "final_response": "..."
      }
    }
  ]
}
```

**Example file**  
<a href="../datasets/agent_evaluation/sample_agent_precaptured_traces.json" download="sample_agent_precaptured_traces.json">sample_agent_precaptured_traces.json</a>

Provide the dataset JSON path (or upload in Colab) and load it with `AgentDatasetLoader`.

In [3]:
try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload your agent dataset JSON file:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    dataset_path = Path(next(iter(uploaded.keys())))
else:
    dataset_path = Path(input("Enter path to agent dataset JSON file: ").strip().strip('"')).expanduser()


### Load the agent dataset

Reads JSON with `AgentDatasetLoader.from_file(...)`. Produces an `AgentDataset` for evaluation.

In [4]:
dataset = AgentDatasetLoader.from_file(dataset_path)
print(f"Dataset loaded from {dataset_path}: {len(dataset.samples)} full sample(s)")


Dataset loaded from C:\Git_Files\Fission\FLOTORCH\floeval\examples\datasets\agent_evaluation\sample_agent_precaptured_traces.json: 1 full sample(s)


## 5. Configure the LLM

Configure the OpenAI-compatible provider used by LLM-judge metrics such as `goal_achievement` and `response_coherence`.

In [5]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model="text-embedding-3-small",
)

## 6. Create and Run the Evaluation

Create `AgentEvaluation` with the dataset, LLM config, and selected metrics, then execute `evaluation.run()`.

In [6]:
evaluation = AgentEvaluation(
    dataset=dataset,
    llm_config=llm_config,
    metrics=["goal_achievement", "response_coherence"],
    default_provider="builtin",
)


In [7]:
results = evaluation.run()
print("Summary:", results.summary)


Summary: {'goal_achievement': 1.0, 'response_coherence': 1.0}


## 7. Inspect Per-Sample Results

Each sample result includes `user_input`, `final_response`, `reference_outcome`, and `metrics` for detailed quality analysis.

In [8]:
for i, sr in enumerate(results.sample_results, start=1):
    print(f"Sample {i}: {sr.get('user_input')}")
    print(f"  Final response: {sr.get('final_response', '')[:80]}...")
    for k, v in sr.get("metrics", {}).items():
        print(f"  {k}: score={v.get('score')}")

Sample 1: Find weather for Paris and summarize.
  Final response: Paris is 18C with light rain today....
  goal_achievement: score=1.0
  response_coherence: score=1.0


## Summary

This notebook demonstrated the evaluation of agents using pre-captured conversation traces.

The key components included:

1. **Agent Sample Structure**: An `AgentDataset` with full traces was loaded from JSON via `AgentDatasetLoader.from_file`.
2. **LLM Configuration**: The OpenAI-compatible provider was configured for LLM-judge metrics.
3. **Evaluation Execution**: The `goal_achievement` and `response_coherence` metrics were run via the builtin provider.
4. **Results Inspection**: The summary and per-sample metrics were accessed through the results object.

This example showcases the workflow for evaluating agents when pre-recorded traces are available.